# ZipVend Assignment

Please complete the assignment by modifying the markdown code (the text cells) and the Python code below. 

**You are required to submit a PDF file for the assignment, not the jupyter notebook itself! The instructions on how to create a PDF file from VSCode are below.**

1. Click on the three dots in the VSCode menu bar as highlighted in the figure below, then click on Export.

 <p align="center">
 <img src="Figures/Export_PDF.png"  width="1800">
 </p>

2. Most likely, if you select PDF, you will encounter some export issues. I'd suggest you export the notebook in HTML format. Open the .html file (which should easily open in your browser) and then export it as a PDF file. 

3. **Make sure that the output of your cells is visible!!!**

--- 
# Part 1: Database discovery with DBeaver
The **data team** has given you a SQLite database (zip_vend.db, located in the data subfolder) with the "static" information about ZipVend: details about ZipVend machines, their locations, and products.

Firstly, connect DBeaver to the zipvend database to explore the data and please respond to the following questions. 

**To edit the text cells, double click on them and edit them! When you are done editing them, press shift + Enter (they should go back to "view mode").**

#####  **Q1. What information does ZipVend keep about its sites, machines, and products? Count how many fields each table uses to describe these entities.**

**location**: Zipvend has 5 fields for location: location_id, site_name, venue_type, borough, postcode_district.

**machine**: Zipvend has 6 fields for machine: machine_id, location_id, install_date, model, segment, is_active.

**product**:Zipvend has 6 fields for product: product_id, sku_name, category, unit_size, cost_price_gbp, base_price_gbp

#####  **Q2. How does ZipVend uniquely identify each site, machine, and product in its catalog (primary keys)?**


**location**: Sites are uniquely identified by the location_id primary key.

**machine**: Machines are uniquely identified by the machine_id primary key.

**product**: Products are uniquely identified by the product_id primary key.


#####  **Q3. How does ZipVend establish links between locations, machines and products ? Check which fields (foreign keys) create those links.**

**location**: No foreign key.

**machine**: Location is linked to machine via the location_id foreign key.

**product**: No foreign key.

#####  **Q4. What types of relationships (established through foreign keys) does this database have?**

The location and machine tables have a one-to-many relationship; one location is associated with multiple machines.


#####  **Q5. Draw a simple diagram (with DBeaver) that shows how ZipVend organizes its catalog: sites, machines, and products. Share it with your notebook so a manager could understand at a glance.**

Please, take a screenshot, move it in the Figures subfolder alongside the other figures and embed it in the jupyter notebook by modifying the markdown code below (edit the src="Figures/ZipVend_logo.png" below so that it points to your file).

 <p align="center">
 <img src="Figures/ZipVend_catalogue.png"  width="500">
 </p>

Now, to automatise the process within your code, you have been asked to explore the database by connecting to it using SQL from Python. 

In [1]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("data/zip_vend.db") # connect to the zip_vend.db database in the data subfolder

#####  **Q6. Show the full list of sites where ZipVend has machines. This is like exporting the company’s location registry.**

In [19]:
query = """
SELECT DISTINCT *
FROM location
"""

df = pd.read_sql_query(query, conn)
df.head(5)

,location_id,site_name,venue_type,borough,postcode_district
0,1,Greenwich Office 1,Office,Greenwich,EC1
1,2,Hackney Office 2,Office,Hackney,NW1
2,3,Hackney Office 3,Office,Hackney,E1
3,4,Greenwich Office 4,Office,Greenwich,SW1
4,5,Southwark Office 5,Office,Southwark,W1


#####  **Q7. List all the products ZipVend offers, their category, and their standard price. This is the product catalog management would review.**

In [3]:
query = """
SELECT DISTINCT sku_name, category, base_price_gbp
FROM product
"""

df = pd.read_sql_query(query, conn)
df.head(5)

,sku_name,category,base_price_gbp
0,SparkFizz 330ml,Drink,1.2
1,SparkFizz 500ml,Drink,1.6
2,ColaMax 330ml,Drink,1.4
3,ColaMax Zero 330ml,Drink,1.4
4,LemonZest 500ml,Drink,1.5


--- 
# PART 2: ZipVend API

Below, you are asked to query the ZipVend API to obtain business insight from the sales data. You can use the documentation of the API available at the /docs endpoint of the API. The BASE_URL of the API and the API_KEY are found in the Assignment brief.

In [ ]:
import requests
from pprint import pprint

# Set the parameters to query the API
BASE_URL = "https://zipvend-api-624076366746.europe-west2.run.app"   
API_KEY = ""
API_headers = {"X-API-Key": API_KEY}

#####  **Q8. How many machines are active vs inactive, and how many locations and products do we have?**

In [17]:
endpoint = '/summary/health'
url = f'{BASE_URL}{endpoint}'
resp = requests.get(url, headers=API_headers)
resp.raise_for_status() # check that the result of the query went well
data = resp.json()
pprint( data )

{'locations': 25, 'machines': {'active': 46, 'inactive': 3}, 'products': 36}


*46 Machines are active while 3 are inactive. There are 25 locations and 36 products*

#####  **Q9. Which reporting weeks are available for this assignment?**

In [18]:
endpoint = '/summary/weeks'
url = f'{BASE_URL}{endpoint}'
r = requests.get(url, headers=API_headers)
r
weeks_info = r.json()

pprint(weeks_info)

{'weeks': [22, 23, 24, 25, 26]}


*Weeks 22 to 26 are available for this assignment*

#####  **Q10. For the latest reporting week, how many units did we sell, what was the gross revenue, how many restock visits, and how many active machines?**

In [7]:
endpoint = '/summary/week/26'
url = f'{BASE_URL}{endpoint}'
r= requests.get(url, headers=API_headers)
r
kpi_latest = r.json()

pprint(kpi_latest)

{'active_machines': 48,
 'restock_visits': 29,
 'revenue_gbp': 10042.25,
 'units_sold': 5822,
 'week_no': 26}


*For reporting week 26, please see the values for the relevant KPIs below:*


*Units sold: 5822*

*Gross revenue: 10042.25 (GBP)*

*Restock visits: 29*

*Active Machines: 48*

#####  **Q11. Did our restocking workload increase or decrease compared with the first week? What about revenue?**

In [8]:
endpoint = '/summary/week/22'
url = f'{BASE_URL}{endpoint}'
r = requests.get(url, headers=API_headers)
r # check that the result of the query went well
kpi_first = r.json()
pprint(kpi_first)

{'active_machines': 47,
 'restock_visits': 39,
 'revenue_gbp': 9554.04,
 'units_sold': 5551,
 'week_no': 22}


*Weekly restocks decreased from week 22 (39 restocks) to week 26 (29 restocks).*
*Weekly revenue increased, however, from 9554.04 to 10042.25* 


#####  **Q12.a Which week delivered the highest revenue?**


In [9]:
kpi_rows = []
for w in [22,23,24,25,26]:
    endpoint = f'/summary/week/{w}'
    url = f'{BASE_URL}{endpoint}'
    r = requests.get(url, headers=API_headers)
    r.raise_for_status # check that the result of the query went well
    pprint(r.json())


{'active_machines': 47,
 'restock_visits': 39,
 'revenue_gbp': 9554.04,
 'units_sold': 5551,
 'week_no': 22}
{'active_machines': 46,
 'restock_visits': 20,
 'revenue_gbp': 9646.49,
 'units_sold': 5503,
 'week_no': 23}
{'active_machines': 46,
 'restock_visits': 19,
 'revenue_gbp': 8922.95,
 'units_sold': 5307,
 'week_no': 24}
{'active_machines': 48,
 'restock_visits': 11,
 'revenue_gbp': 8831.22,
 'units_sold': 5199,
 'week_no': 25}
{'active_machines': 48,
 'restock_visits': 29,
 'revenue_gbp': 10042.25,
 'units_sold': 5822,
 'week_no': 26}


*Week 26 delivered the highest revenue: 10042.25 (GBP)*

In [10]:
# create dataframe from the weekly kpi data for all weeks
kpi_rows = []
for w in [22,23,24,25,26]:
    endpoint = f'/summary/week/{w}'
    url = f'{BASE_URL}{endpoint}'
    r = requests.get(url, headers=API_headers)
    r.raise_for_status() # check that the result of the query went well
    kpi_data = r.json()
    kpi_data['week'] = w
    kpi_rows.append(kpi_data)
kpi_df = pd.DataFrame(kpi_rows)
kpi_df.sort_values(by='revenue_gbp', ascending=False)

,week_no,units_sold,revenue_gbp,restock_visits,active_machines,week
4,26,5822,10042.25,29,48,26
1,23,5503,9646.49,20,46,23
0,22,5551,9554.04,39,47,22
2,24,5307,8922.95,19,46,24
3,25,5199,8831.22,11,48,25


#####  **Q12.b Looking at the trend in revenue vs. restock visits, what would you recommend to improve efficiency in operations?**

*Generally, there is a positive correlation between restock visits and revenue, however week 22 is an outlier. To improve efficiency, Zipvend should optimise restocking by looking at the previous weekly units sold to estimate the number of restocks necessary for the upcoming week* 


# PART 3: Expand your local database
The API also provides endpoints to download the full sales and restock data for all weeks. You are here asked to download the data, expand your local database and perform more advanced sales analysis using SQL.

#####  **Q13. Download the sales data for all weeks and save it in a csv file, in the data subfolder, named "all_sales.csv"**

In [11]:
endpoint = '/files/weekly-sales'
url = f'{BASE_URL}{endpoint}'
r = requests.get(url, headers=API_headers)
r # check that the result of the query went well

# open the file named all_sales.csv in the data subfolder
f = open('data/all_sales.csv', 'wb') # w means write and b means binary; useful format
f.write(r.content) # write the data in the file
f.close()

#####  **Q14. Download the restock data for all weeks and save it in a csv file, in the data subfolder, named "all_restocks.csv"**

In [12]:
endpoint = '/files/weekly-restocks'
url = f'{BASE_URL}{endpoint}'
r = requests.get(url, headers=API_headers)
r.raise_for_status # check that the result of the query went well

f = open('data/all_restocks.csv', 'wb')# open the file named all_sales.csv in the data subfolder
f.write(r.content) # write the data in the file
f.close()

#####  **Q15. Update the database by introducing table Sales and Restocks: specify the primary keys and create relations among all tables by setting foreign keys using DBeaver**

Please thoroughly describe below the process of expanding the database, including a screenshot of the final schema by replacing the zipvend logo as in question 5 (click on Refresh in the Diagram page in DBeaver to automatically adjust the schema diagram).

 <p align="center">
 <img src="Figures/updated_db.png"  width="500">
 </p>

 I expanded the database by adding tables for sales and restocks. For each table, I firstly went to the original database and added a table, then I added a new column with unique numerical values to create a primary key (sales_id and restock_id respectively). Once I renamed the table with their respective names, I clicked save. Now my table was prepared to import the extra data; I clicked import data and added the respective csv files.
 
 With the primary keys already complete, now I had to connect the table with foreign keys. For both sales and restocks, the foreign keys were machine_id for the machine table and product_id for the product table.

---
The engineering team says that the KPI obtained through the API are precomputed but they suspect there could be mistakes. You have been asked to check with direct inspection of the full data by using appropriate SQL queries.

#####  **Q16. Which week had the highest total revenue. Does it agree with what the API gave?**

In [13]:
query = """
SELECT week_no AS Week, SUM(gross_revenue_gbp) AS Total_Revenue
FROM Sales
GROUP BY week_no
ORDER BY Total_Revenue DESC
"""
# get the query in pandas
df = pd.read_sql_query(query, conn)
df.head(5)

,Week,Total_Revenue
0,26,10042.25
1,23,9646.49
2,22,9554.04
3,24,8922.95
4,25,8831.22


Week 26 had the highest gross revenue of 10,042.25 GBP. This agrees with the API (see question 12a).

#####  **Q17. How many machines were active (sold at least 1 unit) in week 26?**

In [14]:
# count machine_ids for each week from sales
query = """
SELECT week_no AS Week, COUNT(DISTINCT machine_id) AS Machines_Active
FROM Sales
GROUP BY Week
"""

# get the query in pandas
pd.read_sql_query(query, conn).head(5)

,Week,Machines_Active
0,22,47
1,23,46
2,24,46
3,25,48
4,26,48


In week 26, 48 machines were active.

#####  **Q18. Which vending machines are installed at Greenwhich University3?**

In [15]:
# join machine and location tables on location id only, get site id from location table
# count machine_id and group by location 
query = """
SELECT site_name AS site, model AS Machine
FROM machine
LEFT JOIN location ON machine.location_id = location.location_id 
WHERE location.site_name = 'Greenwich University 3'
"""
# get the query in pandas
pd.read_sql_query(query, conn)


,site,Machine
0,Greenwich University 3,VM-Combo-2
1,Greenwich University 3,VM-Drink-1


Greenwich University has two machines on site: VM-Combo-2 and VM-Drink-1.

#####  **Q19. Which product category generated the highest total revenue (and how much was this revenue) in the last reporting week?**

In [16]:
query = """
SELECT category AS Product_Category, SUM(gross_revenue_gbp) AS Week_26_Revenue
FROM Sales, product
WHERE week_no = 26 AND Sales.product_id = product.product_id
GROUP BY category
ORDER BY Week_26_Revenue DESC
"""

# get the query in pandas
pd.read_sql_query(query, conn).head(5)

,Product_Category,Week_26_Revenue
0,Energy,4187.84
1,Drink,2478.74
2,Healthy,1855.58
3,Snack,1520.09


In Week 26, Energy was the category with the highest total revenue: £4187.84

#####  **Q20. Designing your own database for a business application**

In the previous parts, you explored and queried an existing database designed for ZipVend’s vending machine operations.

Now, imagine you are the data manager for a different business application of your choice. This could be in retail, finance, healthcare, education, logistics, sports, hospitality, or any other domain that interests you.

Your task:

- Pick a business scenario that would benefit from structured data management.

- Describe the key business questions the company would want to answer with data.

- Propose a database design:
    - Identify the tables you would need.
    - For each table, describe the main columns (attributes) and the primary key.
    - Identify the key relationships between tables.

- Explain why your design works for the chosen business questions!

When writing your answer, focus on clarity and creativity: we want to see how well your database design reflects real business needs.

A family-run educational consultancy must calculate end-user progress every half term (~5-8 weeks) and produce invoices monthly. The business has scaled to increase both customer attainment and retention in recent years, so maintaining data on, and extrapolating insights from, google sheets is inefficient and costly. Transferring data to a database would facilitate more efficient admin  as well as capabilities to perform customer audits.

The key business questions that could be quickly answered by transferring data to a database like Dbeaver are as follows:
What does each client owe at the end of each month?
What is the rate of attendance for each student?
What percentage of clients who are not repeat customers?
What is the relationship between attendance and improvement in attainment?

The data base design would include tables directly transferred from google sheets:
- REGISTRATION FORM (primary key: account reference | attributes: parent email, parent name, student email, student name, student DOB, student SEND requirements, additional parent email, additional parent name, additional parent email, authorised pick up name 1, authorised pick up name 2, english as first language yes/no, )
- REGISTRATION SHEET TABLES (primary key: student name | attributes: parent name, parent email, parent phone number, student email, home address, )
- PROGRESS SHEET TABLE (primary key: student name | attributes: weekly homework submission [binary variable], attainment on half-termly assessments)
- Y1-6 CLASS ATTENDENCE BY CLASS TYPE (primary key: student name | attributes: dates of academic calendar, classes within the week)
- Y7-11 CLASS ATTENDENCE BY CLASS TYPE (primary key: student name | attributes: dates of academic calendar, classes within the week)

The database design would also include new tables for customer auditing:
- KS1-4 CLASSES: (primary key: tutor | attributes: student name, student SEND requirements, student email )
- INVOICE TRACKER: (primary key: parent name | attributes: student name, account reference, half term, date of class, class attended, class price)

I would assigning the following foreign keys to one-to-many relationships: 
1) Account Reference (client database by year group; invoice tracket)
2) Student Name (registration form; invoice tracker; attendance sheet [y1-6 and y7-11], KS1-4 classes)

By managing the database with these foreign keys it would be easy to join tables and to:

 • calculate monthly invoices

 • track monthly changes in revenue so that product lines can be optimised across the academic year

 • shortlist non-repeat clients for push marketing and collection of feedback 
 
 • assess how attendance impacts end-user performance.

